# Treino g1_v3 no RunPod

Deploy do pod (PyTorch 2.8.0 / CUDA 12.8.1, RTX PRO 4500) → abrir o Web Terminal.

- No Web Terminal do pod: `rm -rf /workspace/g1_training && git clone --branch g1_v3 --depth 1 https://github.com/JoaoBornelli/g1_training.git /workspace/g1_training`
- No Jupyter, abra `/workspace/g1_training/g1_v3/container/treino_runpod.ipynb` e use Run All.
- O `rm -rf` apaga só o clone; logs e checkpoints ficam em `/workspace/g1_v3`.
- Para retomar: Start no pod, a mesma linha de clone, RETOMA = 1 na config, Run All.
- Vídeo: com VIDEO = 1 (padrão), o treino grava 20 s a cada 500 iterações em `/workspace/g1_v3/g1_v3/<corrida>/videos/train/`. Na retomada, a contagem recomeça: o primeiro vídeo sai 500 iterações depois do início da sessão.
- Run All para na célula do painel (laço ao vivo). Para usar as células de gráfico livre e de log, interrompa o painel; o treino continua.
- O pod para sozinho no fim do treino.

In [ ]:
# Configuração. Única célula que você precisa editar.
import re

RUN = "poc1"            # nome da corrida: letras, números, '.', '-' e '_'
RETOMA = 0              # 0 = corrida nova; 1 = retoma a última corrida RUN com checkpoint
ITERS = 30000           # iterações desta sessão (na retomada, somam ao início)
NUM_ENVS = 4096
VIDEO = 1               # 1 = grava 20 s de vídeo a cada 500 iterações (a célula 3 instala o EGL); 0 = sem vídeo
TIMEOUT_H = 45          # teto de horas: passou disso, o treino para e o pod para
PRECO_HORA = 0.90       # US$/h, só para a previsão de custo
RAIZ = "/workspace/g1_training"
SAIDA = "/workspace/g1_v3"
LOG = f"{SAIDA}/{RUN}.log"

assert re.fullmatch(r"[A-Za-z0-9._-]+", RUN), "RUN aceita só letras, números, '.', '-' e '_'"
assert RETOMA in (0, 1), "RETOMA é 0 ou 1"
assert VIDEO in (0, 1), "VIDEO é 0 ou 1"
print(f"RUN={RUN}  RETOMA={RETOMA}  ITERS={ITERS}  NUM_ENVS={NUM_ENVS}  VIDEO={VIDEO}")

In [ ]:
# Código: não clona nem faz pull. O clone é feito no Web Terminal (linha da célula de título).
import os, subprocess

assert os.path.isdir(f"{RAIZ}/.git"), (
    f"{RAIZ} não é um clone. No Web Terminal do pod rode: "
    "rm -rf /workspace/g1_training && git clone --branch g1_v3 --depth 1 "
    "https://github.com/JoaoBornelli/g1_training.git /workspace/g1_training")

r = subprocess.run(["git", "-C", RAIZ, "log", "--oneline", "-1"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

In [ ]:
# Instalação: libs do vídeo (se VIDEO = 1), mjlab e matplotlib. Não instala torch: o template já traz o 2.8.
import os, subprocess, sys
import importlib.metadata as md

# Vídeo: o template PyTorch do RunPod não traz o EGL nem o ffmpeg; sem eles o import do mujoco quebra com MUJOCO_GL=egl.
if VIDEO == 1:
    apt = subprocess.run("apt-get update -qq && apt-get install -y -qq libegl1 libopengl0 libgl1 ffmpeg",
                         shell=True, capture_output=True, text=True,
                         env=os.environ | {"DEBIAN_FRONTEND": "noninteractive"})
    print((apt.stdout + apt.stderr)[-800:])
    assert apt.returncode == 0, "apt-get falhou (veja a saída acima); use VIDEO = 0 para treinar sem vídeo"

# mjlab não traz matplotlib nos requisitos, por isso ele entra aqui.
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-warn-conflicts",
                    "mjlab==1.5.3", "matplotlib"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1200:])
assert r.returncode == 0, "pip falhou (veja a saída acima)"

# Se o pip trocou o torch, a CUDA pode ter sumido: confere em subprocesso.
chk = subprocess.run([sys.executable, "-c",
                      "import torch; print(torch.cuda.is_available(), torch.version.cuda)"],
                     capture_output=True, text=True)
print("CUDA (torch):", chk.stdout.strip() or chk.stderr[-400:])
assert chk.stdout.startswith("True"), "torch sem CUDA depois do pip: reinicie o kernel"

for pacote in ["mjlab", "rsl-rl-lib", "torch", "mujoco", "warp-lang", "matplotlib"]:
    print(f"{pacote}: {md.version(pacote)}")

In [ ]:
# Lança o treino solto do kernel: ele sobrevive a fechar o navegador ou o kernel.
import os, subprocess

# a) Se já existe treino rodando, não lança outro.
try:
    vivo = subprocess.run(["pgrep", "-f", "g1_v3.train"], capture_output=True, text=True).stdout.split()
except FileNotFoundError:
    raise RuntimeError("pgrep não existe neste pod: não arrisco lançar outro treino")

if vivo:
    print("JÁ EXISTE treino rodando (PID", " ".join(vivo), "). NÃO lancei outro.")
else:
    env = os.environ | {"RUN": RUN, "RETOMA": str(RETOMA), "ITERS": str(ITERS),
                        "NUM_ENVS": str(NUM_ENVS), "VIDEO": str(VIDEO), "SAIDA": SAIDA}
    # O stop do pod está desligado: na poc3 (06/11) ele apagou o volume /workspace e todos os checkpoints.
    # assert os.environ.get("RUNPOD_POD_ID"), "RUNPOD_POD_ID vazio no kernel: o pod não pararia sozinho no fim. Nada foi lançado."

    # b) Validação sem GPU. Um erro de entrada aqui evita lançar um treino que falha na entrada.
    seco = subprocess.run(["bash", f"{RAIZ}/g1_v3/container/entrada.sh"],
                          env=env | {"SECO": "1"}, capture_output=True, text=True)
    print(seco.stdout, seco.stderr)
    assert seco.returncode == 0, "validação SECO falhou: nada foi lançado"

    # c) Lançamento. Ao terminar (ou estourar TIMEOUT_H), o pod FICA LIGADO e cobra a GPU até ser parado à mão.
    os.makedirs(SAIDA, exist_ok=True)
    log = open(LOG, "ab")
    proc = subprocess.Popen(
        ["bash", "-c", f"timeout {TIMEOUT_H}h bash {RAIZ}/g1_v3/container/entrada.sh"],
                       # f"; runpodctl stop pod $RUNPOD_POD_ID"  # desligado: o stop apagou o volume na poc3 (06/11)
        env=env | {"SECO": "0"}, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True, cwd=RAIZ)
    print("treino lançado, PID", proc.pid)
    print("log:", LOG)
    print("Fechar o navegador ou o kernel NÃO para o treino.")

In [ ]:
# Painel ao vivo. Interromper esta célula NÃO para o treino: ele roda no pod.
import glob, os, re, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def pastas(run):
    """Pastas AAAA-MM-DD_HH-MM-SS_<run> do RUN, em ordem de nome (= ordem de tempo)."""
    padrao = r"\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_" + re.escape(run)
    achadas = glob.glob(f"{SAIDA}/g1_v3/*_{run}")
    return sorted(p for p in achadas if re.fullmatch(padrao, os.path.basename(p)))


def carrega(run):
    """{tag: (steps, valores)} juntando as pastas do RUN, ordenado por step.
    Passo repetido na retomada fica com o valor da pasta mais nova."""
    dados = {}
    for pasta in pastas(run):
        ea = EventAccumulator(pasta, size_guidance={"scalars": 0})
        ea.Reload()
        for tag in ea.Tags()["scalars"]:
            serie = dados.setdefault(tag, {})
            for ev in ea.Scalars(tag):
                serie[ev.step] = ev.value
    saida = {}
    for tag, serie in dados.items():
        steps = sorted(serie)
        saida[tag] = (np.array(steps), np.array([serie[s] for s in steps]))
    return saida


def inicio_da_sessao(run):
    """Primeiro step da pasta mais nova: início da sessão atual (base do alvo na retomada)."""
    ps = pastas(run)
    if not ps:
        return 0
    ea = EventAccumulator(ps[-1], size_guidance={"scalars": 0})
    ea.Reload()
    if "Train/mean_reward" not in ea.Tags()["scalars"]:
        return 0
    return ea.Scalars("Train/mean_reward")[0].step


def tempo_por_it(dados):
    """Collection + learning por iteração, ou None se faltar alguma das duas tags."""
    if "Perf/collection_time" not in dados or "Perf/learning_time" not in dados:
        return None
    cs, cv = dados["Perf/collection_time"]
    ls, lv = dados["Perf/learning_time"]
    lmap = dict(zip(ls.tolist(), lv.tolist()))
    pares = [(s, v + lmap[s]) for s, v in zip(cs.tolist(), cv.tolist()) if s in lmap]
    if not pares:
        return None
    return np.array([p[0] for p in pares]), np.array([p[1] for p in pares])


def linhas(ax, dados, tags, titulo):
    """Desenha as tags num eixo; a legenda usa o nome depois do último '/'."""
    ax.set_title(titulo, fontsize=10)
    ax.grid(alpha=0.3)
    tags = [t for t in tags if t in dados]
    for t in tags:
        s, v = dados[t]
        ax.plot(s, v, lw=1, label=t.split("/")[-1])
    if tags:
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "sem dados ainda", ha="center", va="center", transform=ax.transAxes)


def desenha(dados, alvo=None):
    """Monta a figura do painel. alvo = iteração final esperada (None = desconhecida)."""
    fig, ((a1, a2), (b, d), (c1, c2)) = plt.subplots(3, 2, figsize=(14, 13))
    linhas(a1, dados, ["Train/mean_reward"], "recompensa média")
    linhas(a2, dados, ["Train/mean_episode_length"], "duração do episódio")
    linhas(c1, dados, sorted(t for t in dados if t.startswith("Curriculum/v3/nivel_")), "currículo: nível")
    linhas(c2, dados, sorted(t for t in dados if t.startswith("Curriculum/v3/frac_")), "currículo: fração")
    linhas(d, dados, sorted(t for t in dados if t.startswith("Episode_Metrics/phi_"))[:8],
           "sucesso por tarefa (phi)")

    tempo = tempo_por_it(dados)
    b.set_title("tempo por iteração (s)", fontsize=10)
    b.grid(alpha=0.3)
    if tempo is None:
        b.text(0.5, 0.5, "aguardando tempo por iteração", ha="center", va="center", transform=b.transAxes)
    else:
        s, v = tempo
        b.plot(s, v, lw=1)
        atual = int(dados["Train/mean_reward"][0][-1]) if "Train/mean_reward" in dados else int(s[-1])
        media = v[-20:].mean()
        txt = f"iteração atual: {atual}\ns/it médio (20 últ.): {media:.2f}"
        if alvo is not None:
            rest = max(alvo - atual, 0)
            horas = rest * media / 3600
            txt += (f"\nalvo: {alvo}\nits restantes: {rest}"
                    f"\nprevisão: {horas:.1f} h | USD {horas * PRECO_HORA:.2f} (USD {PRECO_HORA}/h)")
        b.text(0.02, 0.97, txt, transform=b.transAxes, va="top", family="monospace", fontsize=9,
               bbox=dict(facecolor="white", alpha=0.8))
    fig.tight_layout()
    return fig


# Laço ao vivo: a cada 60 s redesenha. KeyboardInterrupt (Interromper) sai em silêncio;
# interromper a célula não para o treino.
try:
    while True:
        dados = carrega(RUN)
        alvo = (inicio_da_sessao(RUN) + ITERS) if RETOMA == 1 else ITERS
        clear_output(wait=True)
        vivo = subprocess.run(["pgrep", "-f", "g1_v3.train"], capture_output=True).returncode == 0
        print(time.strftime("%H:%M:%S"), "| processo do treino vivo:", "sim" if vivo else "NÃO")
        if not dados:
            print(f"aguardando o primeiro log em {SAIDA}/g1_v3/*_{RUN}")
        else:
            fig = desenha(dados, alvo)
            plt.show()
            plt.close(fig)
        time.sleep(60)
except KeyboardInterrupt:
    pass

In [ ]:
# Gráfico livre: todas as tags que contêm FILTRO (sem diferenciar maiúsculas).
import matplotlib.pyplot as plt

FILTRO = "terreno"
dados = carrega(RUN)
tags = sorted(t for t in dados if FILTRO.lower() in t.lower())
print(f"tags com '{FILTRO}': {len(tags)}")
print("\n".join(tags) or "(nenhuma)")
if tags:
    if len(tags) > 12:
        print("só as 12 primeiras são desenhadas")
    tags = tags[:12]
    n_lin = -(-len(tags) // 3)
    fig, eixos = plt.subplots(n_lin, 3, figsize=(14, 3.2 * n_lin), squeeze=False)
    for ax, tag in zip(eixos.flat, tags):
        s, v = dados[tag]
        ax.plot(s, v, lw=1)
        ax.set_title(tag, fontsize=8)
        ax.grid(alpha=0.3)
    for ax in list(eixos.flat)[len(tags):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

# Todas as tags, agrupadas pelo prefixo antes do primeiro "/".
print("\nTodas as tags disponíveis:")
grupos = {}
for t in dados:
    grupos.setdefault(t.split("/")[0], []).append(t)
for g in sorted(grupos):
    print(f"\n{g} ({len(grupos[g])})")
    for t in sorted(grupos[g]):
        print("  ", t)
if not dados:
    print("(sem dados ainda)")

In [ ]:
# Últimas 40 linhas do log do treino.
import os, subprocess

if os.path.exists(LOG):
    print(subprocess.run(["tail", "-n", "40", LOG], capture_output=True, text=True).stdout)
else:
    print("ainda sem log:", LOG)

## Acompanhar e baixar

- **TensorBoard:** em **Connect → HTTP 6006** (o entrada.sh sobe o TensorBoard na porta 6006).
- **Checkpoints:** navegador de arquivos do Jupyter em `/workspace/g1_v3/g1_v3/`.
- **Vídeos:** `/workspace/g1_v3/g1_v3/<corrida>/videos/train/` (um .mp4 a cada 500 iterações).
- **Terminate** zera a cobrança e apaga tudo: baixe os checkpoints antes.